# Dynamic World Playground

This notebook is meant as playground to get familiar with [Dynamic World](https://www.dynamicworld.app/) and aggregate code spinets from different tutorials.

In [ ]:
import ee
import geemap
import os
from dotenv import load_dotenv
import numpy as np
load_dotenv();

To run this project on your machine, you need to authenticate yourself into [Earth Engine](https://developers.google.com/earth-engine/guides/python_install). You should have permissions to access a [Google Cloud Earth Engine project]((https://console.cloud.google.com/earth-engine/welcome)).

In [ ]:
# Initialize the Earth Engine module.
ee.Initialize(project = os.getenv("PROJECT_ID"))

## Data Types
Dynamic World is an `ee.ImageCollection` which is a collection of `ee.Image`. An image is a form of raster data, it is composed of one or more bands. Dynamic World Bands:

| **Index** | **Band Name** | **Description** | **Data Type** | **Range** |
| --- | --- | --- | --- | --- |
| 0 | water (blue) | Estimated probability of complete coverage by water. | double | [0, 1) |
| 1 | trees (dark green) | Estimated probability of complete coverage by trees. | double | [0, 1) |
| 2 | grass (green) | Estimated probability of complete coverage by grass. | double | [0, 1) |
| 3 | flooded_vegetation (greyish blue) | Estimated probability of complete coverage by flooded vegetation. | double | [0, 1) |
| 4 | crops (orange) | Estimated probability of complete coverage by crops. | double | [0, 1) |
| 5 | shrub_and_scrub  (yellow) | Estimated probability of complete coverage by shrub and scrub. | double | [0, 1) |
| 6 | built  (red) | Estimated probability of complete coverage by built area. | double | [0, 1) |
| 7 | bare (grey) | Estimated probability of complete coverage by bare ground. | double | [0, 1) |
| 8 | snow_and_ice (purple) | Estimated probability of complete coverage by snow and ice. | double | [0, 1) |
| 9 | label | Index of the band with the highest estimated probability. | unsigned byte | [0, 8] |

Images coming from Dynamic World have the following properties. 

| Name | Type | Description |
| --- | --- | --- |
| dynamicworld_algorithm_version | STRING | The version string uniquely identifying the Dynamic World model and inference process used to produce the image. |
| qa_algorithm_version | STRING | The version string uniquely identifying the cloud masking process used to produce the image. |

We can filter time for location and time.

In [ ]:
# Define date and region
START = ee.Date('2021-04-02')
END = START.advance(1, 'year')
LOCATION = ee.Geometry.Point(20.6729, 52.4305)

# define filter
col_filter = ee.Filter.And(
    ee.Filter.bounds(LOCATION), 
    ee.Filter.date(START, END),
)

# Get relevant DW images
dw_col = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(col_filter)

# Inspect first image
# geemap.image_props(dw_col.first())
display(dw_col.first())

DW images are generated based on Sentinel-2 images, which are stored in a different collection. We can link the images from the two collections. This will result in a new collection, in which the images will contain both the classification values coming from DW and the original satellite image. Since DW classifications are only available for scenes with less than 35% cloud cover, we can filter Sentinel-2 image collection for images with `'CLOUDY_PIXEL_PERCENTAGE'` smaller than 35%.

In [ ]:
cloud_filter = ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 35)

s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filter(cloud_filter)
linked_col = dw_col.linkCollection(s2_col, s2_col.first().bandNames())

### Map Visualization
We use python’s `geemap` package for interactive geospatial analysis and visualization. The map object is based on ipyleafletmap object.

In [ ]:
# Create map
map = geemap.Map(scroll_wheel_zoom = False)
map.set_center(20.6729, 52.4305, 12)

DW_CLASS = {
    'water' : '419bdf', # blue
    'trees' : '397d49', # green
    'grass' : '88b053', # light green
    'flooded_vegetation' : '7a87c6', # greyish blue
    'crops' : 'e49635', # orange
    'shrub_and_scrub' : 'dfc35a', # yellow
    'built' : 'c4281b', # red
    'bare' : 'a59b8f', # grey
    'snow_and_ice' : 'b39fe1', # purple
}

DW_VIZ_PARAMS = {
  'min' : 0,
  'max' : 8,
  'palette' : list(DW_CLASS.values()),
  'bands' : 'label'
}

# B2, B3, B4 are the optical visualization bands, 
# min defines the values that should be mapped to RGB(0) and max the values that should be mapped to RGB(255)
S2_VIZ_PARAMS = {
  'min' : 0,
  'max' : 3000,
  'bands' : ['B4', 'B3', 'B2']
}


In [ ]:
img = linked_col.first()

# Add Sentinel-2 layer 
map.add_layer(
    img,
    S2_VIZ_PARAMS, 
    'Sentinel-2 L1C',
)
# Add DW classification layer
map.add_layer(
    img,
    DW_VIZ_PARAMS,
    'Dynamic World',
)
map

### Hillshade Map
Hillshade maps, also known as terrain maps, offer a three-dimensional portrayal of landscapes on a two-dimensional surface. We can use hillshade maps to visually represent the confidence of a class prediction. The following technique uses a hillshade computed from a probability image containing the value of the highest probability at each pixel (Top-1 Probability).  Higher confidence of a class membership represents higher elevation and lower confidence represents lower elevation. The resulting hillshade rendering contains ridges and valleys showing the varying class confidence across the landscape—revealing features not normally visible with discretize visualization.

In [ ]:
img = linked_col.first()

map_hillshade = geemap.Map(scroll_wheel_zoom = False)
map_hillshade.centerObject(img, 12)

# For each pixel get the highest probability value among all class probabilities
top1_prob = img.select(list(DW_CLASS.keys())).reduce(ee.Reducer.max())

# Create a hillshade of the most likely class probability on [0, 1]
# The algorithm expects values in meters so we convert probabilities by multiplying them by 100
# The algorithm returns an RGB image, we convert it back to probabilities by diving by 255
top1_prob_hillshade = ee.Terrain.hillshade(top1_prob.multiply(100)).divide(255)

# Combine RGB information and hillshade
dw_rgb_hillshade = img.visualize(**DW_VIZ_PARAMS).divide(255).multiply(top1_prob_hillshade)

# Display the Dynamic World visualization with the source Sentinel-2 image.
map_hillshade.add_layer(
    img,
    S2_VIZ_PARAMS, 
    'Sentinel-2 L1C',
)
map_hillshade.add_layer(
    dw_rgb_hillshade,
    {min: 0, max: 0.65},
    'Dynamic World V1 - label hillshade',
)
map_hillshade

## Time-Series
We can aggregate the estimated distribution of the LULC classes over the time period by selecting the most frequently occurring class label for each pixel during the year or by using the hillshade, creating a mean composite that has the average pixel-wise probability for each band across images captured at different times

In [ ]:
# geemap.image_props(most_frequent_class)

In [ ]:
map_year = geemap.Map(scroll_wheel_zoom = False)
map_year.set_center(20.6729, 52.4305, 12)

# Most Frequently Occurring Label
most_frequent_class = dw_col.select('label').reduce(ee.Reducer.mode().setOutputs(['label']))

# Mean Probability HillShade
# Get the mean for each band in each pixel
# The mean reducer changes the CSR so we need to reproject back to DW EPSG
prob_mean = dw_col.select(list(DW_CLASS.keys())).reduce(ee.Reducer.mean()).reproject(crs='EPSG:3857', scale=10)
prob_mean = prob_mean.rename(list(DW_CLASS.keys()))

# Find the band with the highest mean probability in each pixel
array_img = prob_mean.toArray()
argmax = array_img.arrayArgmax()
highest_band_index = argmax.arrayGet(0)
highest_band_index = highest_band_index.rename(['label'])

prob_mean = prob_mean.addBands(highest_band_index)

# Find the value of the highest probability
top1_prob_year = prob_mean.select(list(DW_CLASS.keys())).reduce(ee.Reducer.max())
top1_prob_hillshade_year = ee.Terrain.hillshade(top1_prob_year.multiply(100)).divide(255)
dw_rgb_hillshade_year = prob_mean.visualize(**DW_VIZ_PARAMS).divide(255).multiply(top1_prob_hillshade_year)

map_year.add_layer(
    dw_rgb_hillshade_year,
    {min: 0, max: 0.65},
    'Dynamic World - year',
)
map_year

We can also add a time_slider to the map to interactively watch the classifications changing over time.

In [ ]:
map.add_time_slider(dw_col, DW_VIZ_PARAMS, time_interval=1)
map

## Changes